# Interpret region-region interactions

The ``interpret_region_region_interactions`` command uses the pre-trained ChromBERT model or fine-tuned ChromBERT to infer region-region interaction on user-specified enhancer regions.

**Note**: The remaining examples show command-line usage only (bash).

For the Python API, see [`examples/api/interpret_region_region_interactions.ipynb`](../api/interpret_region_region_interactions.ipynb).

For Singularity, see [`singularity_use.ipynb`](./singularity_use.ipynb) for `singularity exec` with chrombert-tools.

### infer region-region interactions (enhancer-promoter loop; only by pretrained chrombert)

In [1]:
import pandas as pd
!mkdir -p 'output_infer_ep'
data = pd.read_csv("../data/hESC_GSM2386582_ATAC.bed",sep='\t',header=None)
data
data[data[0]=='chr1'].to_csv('./output_infer_ep/hESC_chr1.bed',sep='\t',header=False,index=False)

In [4]:
%%bash
chrombert-tools interpret_region_region_interactions \
    --region './output_infer_ep/hESC_chr1.bed' \
    --odir "./output_infer_ep" \
    --genome "hg38" \
    --resolution "1kb"


Region summary - total: 5262, overlapping with ChromBERT: 5490 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 33
Finished!
Enhancer-promoter style pairs saved to: ./output_infer_ep/tss_region_pairs_cos.tsv


In [5]:
# infer enhancer-promoter loop
# cos_sim: cosine similarity between the enhancer region embedding and the gene promoter (TSS) region embedding; higher values indicate a more likely enhancer–promoter loop.
tss_region_pairs_cos = pd.read_csv("output_infer_ep/tss_region_pairs_cos.tsv",sep='\t')
tss_region_pairs_cos.head()


,chrom,gene_id,gene_name,tss,tss_build_region_index,distal_region_start,distal_region_end,distal_region_build_region_index,dist,dist_bin,cos_sim
0,chr1,ENSG00000278267,MIR6859-1,17436,2,10000,11000,0,-6436,-2,0.609375
1,chr1,ENSG00000278267,MIR6859-1,17436,2,181000,182000,39,163564,37,0.738281
2,chr1,ENSG00000227232,WASH7P,29570,3,10000,11000,0,-18570,-3,0.599121
3,chr1,ENSG00000227232,WASH7P,29570,3,181000,182000,39,151430,36,0.706543
4,chr1,ENSG00000243485,MIR1302-2HG,29554,3,10000,11000,0,-18554,-3,0.599121


In [6]:
tss_region_pairs_cos.query("gene_name == 'RNVU1-15'").sort_values(by='cos_sim',ascending=False)

,chrom,gene_id,gene_name,tss,tss_build_region_index,distal_region_start,distal_region_end,distal_region_build_region_index,dist,dist_bin,cos_sim
47104,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144546000,144547000,99004,133424,79,0.966797
47105,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144551000,144552000,99008,138424,83,0.910645
47106,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144560000,144561000,99015,147424,90,0.794922
47101,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144461000,144462000,98949,48424,24,0.699219
47100,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144419000,144420000,98930,6424,5,0.688477
47103,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144524000,144525000,98985,111424,60,0.651367
47107,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144567000,144568000,99021,154424,96,0.431885
47102,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144490000,144491000,98966,77424,41,0.307373


### infer enhancer-promoter interactions (celltype-specific fine-tuned model)

In [7]:
import pandas as pd
!mkdir -p 'output_infer_ep_myoblast_specific'
data = pd.read_csv("../data/myoblast_ENCFF647RNC_peak.bed",sep='\t',header=None)
data
data[data[0]=='chr1'].to_csv('./output_infer_ep_myoblast_specific/myoblast_chr1.bed',sep='\t',header=False,index=False)

In [9]:
import glob
ft_ckpt_dir = "../api/output_cell_specific_emb_train/train/**/*.ckpt"

ft_ckpt = glob.glob(ft_ckpt_dir, recursive=True)[0]
ft_ckpt

'../api/output_cell_specific_emb_train/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=176.ckpt'

In [17]:
!export CUDA_VISIBLE_DEVICES=1
!chrombert-tools interpret_region_region_interactions \
    --region '../data/myoblast_ENCFF647RNC_peak_100.bed' \
    --odir "./output_infer_ep_myoblast_specific" \
    --genome "hg38" \
    --resolution "1kb" \
    --ft-ckpt $ft_ckpt \
    --batch-size 64


Region summary - total: 100, overlapping with ChromBERT: 101 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 0
Your supervised_file does not contain the 'label' column. Please verify whether ground truth column ('label') is required. If it is not needed, you may disregard this message.
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Loading checkpoint from ../api/output_cell_specific_emb_train/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=176.ckpt
Loading from pl module, remove prefix 'model.'
Loading from pl module, replace 'pretrain_model' with 'pretrain_model.chrombert'
Loaded 111/111 parameters
100%|█████████████████████████████████████████| 792/792 [13:05<00:00,  1.01it/s]
Finished!
Enhancer-promoter style pairs saved to: ./output_infer_ep_myoblast_specific/tss

In [18]:
# infer enhancer-promoter loop
# cos_sim: cosine similarity between the enhancer region embedding and the gene promoter (TSS) region embedding; higher values indicate a more likely enhancer–promoter loop.
tss_region_pairs_cos_myoblast = pd.read_csv("output_infer_ep_myoblast_specific/tss_region_pairs_cos.tsv",sep='\t')
tss_region_pairs_cos_myoblast.head()


,chrom,gene_id,gene_name,tss,tss_build_region_index,distal_region_start,distal_region_end,distal_region_build_region_index,dist,dist_bin,cos_sim
0,chr1,ENSG00000278267,MIR6859-1,17436,2,180000,181000,38,162564,36,0.807517
1,chr1,ENSG00000278267,MIR6859-1,17436,2,181000,182000,39,163564,37,0.742019
2,chr1,ENSG00000278267,MIR6859-1,17436,2,182000,183000,40,164564,38,0.962093
3,chr1,ENSG00000278267,MIR6859-1,17436,2,191000,192000,46,173564,44,0.624105
4,chr1,ENSG00000227232,WASH7P,29570,3,180000,181000,38,150430,35,0.753813


In [ ]:
tss_region_pairs_cos.query("gene_name == 'RNVU1-15'").sort_values(by='cos_sim',ascending=False)

,chrom,gene_id,gene_name,tss,tss_build_region_index,distal_region_start,distal_region_end,distal_region_build_region_index,dist,dist_bin,cos_sim
242161,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144546000,144547000,99004,133424,79,0.974380
242160,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144546000,144547000,99004,133424,79,0.974380
242165,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144552000,144553000,99009,139424,84,0.949564
242164,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144551000,144552000,99008,138424,83,0.934053
242166,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144560000,144561000,99015,147424,90,0.799996
242158,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144524000,144525000,98985,111424,60,0.676938
242138,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144413000,144414000,98926,424,1,0.670687
242163,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144550000,144551000,99007,137424,82,0.632238
242139,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144419000,144420000,98930,6424,5,0.625515
242140,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144419000,144420000,98930,6424,5,0.625515
